# Synthetic control and IV in potential outcomes: practice notebook

**Causal Analytics · Asynchronous module** · Simon Business School · Prof. Tianli Xia

Synthetic California with placebos; compliance types, LATE, and complier means from the JTPA experiment; the wellness program's compliers vs. never-takers; and a latent-index simulation. Exercises are marked `# YOUR CODE HERE`.

**In Colab:** *File → Save a copy in Drive* first. Run cells with **Shift+Enter**.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SITE = "https://tianlixia.com/causal-analytics/A6/"   # the module's website, e.g. "https://<user>.github.io/<repo>/A6/" (filled in when the site is published)

def load(name):
    for p in [Path(name), Path("data") / name, Path("web/data") / name]:
        if p.exists():
            return pd.read_csv(p)
    return pd.read_csv(SITE + "data/" + name)

smoking = load("smoking.csv")
jtpa = load("jtpa.csv")
well = load("illinois_wellness_employer.csv")
print(smoking.shape, jtpa.shape, well.shape)

## Part A. Synthetic California

Prop 99 started in 1989. The donor pool is the 38 states without large tobacco programs. We choose non-negative weights that sum to one and best match California's cigarette sales from 1970 to 1988 (outcomes only; the published study also matches income, price, beer, and youth share).

In [ ]:
Y = smoking.pivot(index="year", columns="state", values="cigsale")
years = Y.index.to_numpy()
pre = years <= 1988

def project(v):                          # closest point with w >= 0 and sum(w) = 1
    u = np.sort(v)[::-1]; c = np.cumsum(u)
    k = np.nonzero(u * np.arange(1, len(v) + 1) > c - 1)[0][-1]
    return np.maximum(v - (c[k] - 1) / (k + 1), 0)

def synth(y1, Y0, iters=3000):           # accelerated projected gradient on the pre-period outcomes
    A, b = Y0[pre], y1[pre]
    L = np.linalg.norm(A, 2) ** 2
    w = z = np.ones(A.shape[1]) / A.shape[1]; t = 1.0
    for _ in range(iters):
        w_new = project(z - A.T @ (A @ z - b) / L)
        t_new = (1 + np.sqrt(1 + 4 * t * t)) / 2
        z, w, t = w_new + (t - 1) / t_new * (w_new - w), w_new, t_new
    return w

def gap(state, data=Y):
    others = data.drop(columns=state)
    w = synth(data[state].to_numpy(), others.to_numpy())
    return data[state] - others.to_numpy() @ w, pd.Series(w, index=others.columns)

g, w = gap("California")
print(w[w > 0.01].round(3).sort_values(ascending=False))
print(f"pre-1989 miss {np.sqrt((g[pre] ** 2).mean()):.2f} packs; average gap 1989-2000 {g[~pre].mean():.1f}")
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(years, Y["California"], label="California")
ax.plot(years, Y["California"] - g, "--", label="synthetic California")
ax.axvline(1988.5, ls=":", color="k"); ax.legend(); ax.set_ylabel("packs per capita"); plt.show()

### A2. Placebos in space: a permutation p-value

In [ ]:
ratios, gaps = {}, {}
for s in Y.columns:
    gs, _ = gap(s)
    gaps[s] = gs
    ratios[s] = np.sqrt((gs[~pre] ** 2).mean() / (gs[pre] ** 2).mean())
r = pd.Series(ratios).sort_values(ascending=False)
print(r.head(5).round(1))
print("California's rank:", list(r.index).index("California") + 1, "of", len(r), "-> p =", round((list(r.index).index("California") + 1) / len(r), 3))
fig, ax = plt.subplots(figsize=(8, 3.5))
for s, gs in gaps.items():
    ax.plot(years, gs, color="red" if s == "California" else "lightgrey", lw=2.5 if s == "California" else 1)
ax.axvline(1988.5, ls=":", color="k"); ax.set_ylabel("gap (packs)"); plt.show()

**Exercise A.** (1) Leave-one-out: drop each donor with weight above 0.05 from the pool in turn and recompute California's average 1989–2000 gap. (2) Backdate: pretend the program started in 1981 (match 1970–80 only) and look at the gap in 1981–88. What do you conclude?

In [ ]:
# YOUR CODE HERE

## Part B. IV in potential outcomes: the JTPA experiment

`assignmt` = offered training at random (Z), `training` = trained (D), `earnings` = 30-month earnings (Y).

In [ ]:
Z, D, Yj = jtpa.assignmt.to_numpy(), jtpa.training.to_numpy(), jtpa.earnings.to_numpy()

def compliance(Z, D, Y):
    p1, p0 = D[Z == 1].mean(), D[Z == 0].mean()
    pc = p1 - p0
    return dict(always=p0, compliers=pc, never=1 - p1,
                ITT=Y[Z == 1].mean() - Y[Z == 0].mean(),
                LATE=(Y[Z == 1].mean() - Y[Z == 0].mean()) / pc,
                y1_compliers=((Y * D)[Z == 1].mean() - (Y * D)[Z == 0].mean()) / pc,
                y0_compliers=((Y * (1 - D))[Z == 0].mean() - (Y * (1 - D))[Z == 1].mean()) / pc,
                y0_never=Y[(Z == 1) & (D == 0)].mean(),
                as_treated=Y[D == 1].mean() - Y[D == 0].mean())

pd.Series(compliance(Z, D, Yj)).round(3)

Two-stage least squares gives the same number as the Wald ratio: the covariance ratio Cov(Z, Y) / Cov(Z, D).

In [ ]:
print(f"Cov(Z,Y)/Cov(Z,D) = {np.cov(Z, Yj)[0, 1] / np.cov(Z, D)[0, 1]:,.0f}")

**Exercise B.** Repeat `compliance` for men and women separately (`sex` = 1 for men). Are the LATEs similar? Are the shares of compliers similar?

In [ ]:
# YOUR CODE HERE

## Part C. What compliers look like: the wellness program

One-sided non-compliance (only those offered can participate), so there are no always-takers.

In [ ]:
for y in ["gym_pre", "gym_post"]:
    d = well.dropna(subset=[y])
    c = compliance(d.offered.to_numpy(), d.participated.to_numpy(), d[y].to_numpy())
    print(f"{y}: compliers Y1 {c['y1_compliers']:.2f}, Y0 {c['y0_compliers']:.2f}; never-takers Y0 {c['y0_never']:.2f}; LATE {c['LATE']:.2f}")

## Part D. A latent-index simulation: whose effect?

Everyone has a resistance `v` (uniform), takes the treatment if `g0 + g1*Z > v`, and gains `5 + slope*(0.5 - v)`.

In [ ]:
rng = np.random.default_rng(0)
def simulate(g0=0.2, g1=0.3, slope=6, n=200_000):
    v = rng.uniform(size=n); Zs = rng.integers(0, 2, n)
    Ds = (g0 + g1 * Zs > v).astype(int)
    gain = 5 + slope * (0.5 - v)
    Y0s = rng.normal(10, 2, n); Ys = Y0s + Ds * gain
    wald = (Ys[Zs == 1].mean() - Ys[Zs == 0].mean()) / (Ds[Zs == 1].mean() - Ds[Zs == 0].mean())
    complier = (v > g0) & (v < g0 + g1)
    return dict(wald=wald, LATE=gain[complier].mean(), ATT=gain[Ds == 1].mean(), ATE=gain.mean())

pd.Series(simulate()).round(2)

**Exercise D.** Make the instrument stronger (`g1=0.6`) and weaker (`g1=0.1`). How does the LATE move, and why? What happens with `slope=0`?

In [ ]:
# YOUR CODE HERE